> **한눈에 보기**
> - LangSmith: `.env`에 `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` 등을 넣으면 LLM 호출 기록이 사이트에 남는다
> - 호출 코드는 평소와 같다 (추적은 환경변수로 켜고 끔)
> - `@tool` + `create_agent(model, tools)`: 모델이 필요할 때 도구를 스스로 호출하는 에이전트
> - `langchain_teddynote.logging.langsmith("프로젝트", set_enable=...)`로 코드에서 추적 켜고 끄기

## 1. 준비: .env에 LangSmith 키 넣기

In [1]:
!pip --version

pip 26.2.1 from D:\moon0902\hanwha_0902\ex0914\.venv\Lib\site-packages\pip (python 3.12)



In [5]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [ ]:
# !pip install -U langchain langchain-openai

In [2]:
from dotenv import load_dotenv
import os

# .env 파일 로드 (override=False가 기본값이므로 시스템 환경변수가 우선순위를 가집니다)
load_dotenv()

# 정상 로드 확인 테스트
print("OpenAI 키:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH 키:", os.getenv("LANGSMITH_API_KEY")[:8] + "...")
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

OpenAI 키: sk-proj-...
LANGSMITH 키: lsv2_pt_...
LangSmith 프로젝트: proj0914


**📝 정리** — 키는 앞 8글자(`sk-proj-`, `lsv2_pt_`처럼 키 종류를 나타내는 부분)만 확인했다. 키 전체가 출력에 남으면 노트북과 함께 깃허브에 올라간다.

## 2. 평소처럼 호출하면 LangSmith에 기록된다

In [7]:
# 코드는 평소와 같다. LangSmith 추적 환경변수가 켜져 있으면 이 호출이 자동으로 기록된다
from langchain_openai import ChatOpenAI 

llm = ChatOpenAI(model="gpt-4o-mini")
question = "일본의 수도는 어디인가요?"
response = llm.invoke(question)
print(response.content)

일본의 수도는 도쿄(Tokyo)입니다. 도쿄는 일본의 정치, 경제, 문화의 중심지로 알려져 있습니다.


## 3. 도구(@tool)와 에이전트

In [3]:
# @tool: 함수를 에이전트가 쓸 수 있는 도구로 만든다 (docstring이 도구 설명)
# create_agent: 모델 + 도구 목록으로 에이전트 생성
# 모델이 질문을 보고 get_weather를 부를지 스스로 판단한다
from langchain_openai import ChatOpenAI 
from langchain_core.tools import tool 
from langchain.agents import create_agent 

# 1. Tool 정의
@tool 
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_agent(model=llm, tools=[get_weather])

result = agent.invoke({"messages": [("user", "San Francisco 날씨 어때?")]})
print(result["messages"][-1].content)

샌프란시스코의 날씨는 항상 맑습니다!


**📝 정리** — 실행 결과 "샌프란시스코의 날씨는 항상 맑습니다!"는 `get_weather`가 돌려준 `It's always sunny in ...`을 모델이 한국어로 바꿔 답한 것이다. 즉 모델이 도구를 실제로 호출했다는 뜻이다.

## 4. langchain_teddynote로 추적 켜고 끄기

In [8]:
# 출력의 ^C는 예전에 설치하다 중단한 흔적 (지금은 주석이라 실행되지 않음)
# !pip install langchain_teddynote

^C


In [ ]:
from langchain_teddynote import logging

# 랭스미스 추적하기
logging.langsmith("proj0914", set_enable=True)

LangSmith 추적을 시작합니다.
[프로젝트명]
proj0914


In [ ]:
# 랭스미스 추적하지 않기
logging.langsmith("proj0914", set_enable=False)

## 🔁 복습 질문

<details>
<summary>1. LangSmith를 쓰면 무엇을 볼 수 있나?</summary>

호출마다 입력 프롬프트, 출력, 걸린 시간, 토큰 수, (에이전트라면) 도구 호출 과정까지 사이트에서 볼 수 있다.
</details>

<details>
<summary>2. 에이전트가 도구를 썼는지 어떻게 알 수 있나?</summary>

답에 도구의 반환값(`always sunny`) 내용이 들어 있고, `result["messages"]` 중간에 도구 호출 메시지가 있다. LangSmith에서도 확인할 수 있다.
</details>
